In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("insurance_encoded.csv")

In [3]:
df.head()

,age,sex,bmi,children,smoker,charges,premium_category,smoker_obese,region_northeast,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,high,0,0,0,0,1
1,18,1,33.770,1,0,1725.55230,low,0,0,0,1,0
2,28,1,33.000,3,0,4449.46200,low,0,0,0,1,0
3,33,1,22.705,0,0,21984.47061,high,0,0,1,0,0
4,32,1,28.880,0,0,3866.85520,low,0,0,1,0,0


In [4]:
from sklearn.preprocessing import LabelEncoder

le2 = LabelEncoder()
df["premium_category"] = le2.fit_transform(df["premium_category"])
print(df["premium_category"].value_counts())
print(le2.classes_)
mapping = {0: "high", 1: "low", 2: "medium"}
print(mapping)

premium_category
2    621
1    359
0    358
Name: count, dtype: int64
['high' 'low' 'medium']
{0: 'high', 1: 'low', 2: 'medium'}


In [5]:
X = df.drop(["charges","premium_category"],axis = 1)
y = df["premium_category"]

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1070, 10)
X_test: (268, 10)
y_train: (1070,)
y_test: (268,)


In [7]:
#importing models 
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
models = {
    "Logistic Regression" : LogisticRegression(max_iter = 1000),
    "KNN" : KNeighborsClassifier(),
    "SVM" : SVC(),
    "Naive Bayes" : GaussianNB(),
    "Decision Tree" : DecisionTreeClassifier(),
    "Random Forest" : RandomForestClassifier(),
    "Gradient Boosting Classifier" : GradientBoostingClassifier()
}

In [10]:
for name , model in models.items():
    model.fit(X_train_scaled,y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test , y_pred)
    print(f"Accuracy of the {name} is: {accuracy}")

Accuracy of the Logistic Regression is: 0.8955223880597015
Accuracy of the KNN is: 0.8805970149253731
Accuracy of the SVM is: 0.9104477611940298
Accuracy of the Naive Bayes is: 0.753731343283582
Accuracy of the Decision Tree is: 0.8283582089552238
Accuracy of the Random Forest is: 0.8992537313432836
Accuracy of the Gradient Boosting Classifier is: 0.9029850746268657


In [11]:
#pre pruning of the decision tree 
dt_pre =  DecisionTreeClassifier(max_depth = 5 , min_samples_split = 10 , min_samples_leaf = 5)
dt_pre.fit(X_train_scaled , y_train)
y_pred_pre = dt_pre.predict(X_test_scaled)
accuracy = accuracy_score(y_test , y_pred_pre)
print(f"Accuracy of decision tree is : {accuracy}")

Accuracy of decision tree is : 0.8992537313432836


In [12]:
#post pruning of decision tree
dt_post = DecisionTreeClassifier(ccp_alpha=0.01)
dt_post.fit(X_train_scaled, y_train)
y_pred_post = dt_post.predict(X_test_scaled)
accuracy = accuracy_score(y_test , y_pred_post)
print(f"Accuracy of decision tree is : {accuracy}")

Accuracy of decision tree is : 0.8917910447761194


In [13]:
from sklearn.ensemble import BaggingClassifier, VotingClassifier, StackingClassifier , AdaBoostClassifier
from xgboost import XGBClassifier

In [14]:
#bagging 
bagging = BaggingClassifier(n_estimators = 100 , random_state = 42)
bagging.fit(X_train_scaled , y_train)
bagging.predict(X_test_scaled)
accuracy_score(y_test , bagging.predict(X_test_scaled))

0.8992537313432836

In [15]:
#ada boosting
boosting = AdaBoostClassifier(n_estimators = 100 , random_state = 42)
boosting.fit(X_train_scaled , y_train)
boosting.predict(X_test_scaled)
accuracy_score(y_test , boosting.predict(X_test_scaled))

0.8917910447761194

In [16]:
#voting
voting = VotingClassifier(
    estimators = [
        ("lr" , LogisticRegression(max_iter = 1000)),
        ("svm" , SVC()),
        ("dt" , DecisionTreeClassifier())
    ]
)
voting.fit(X_train_scaled , y_train)
voting.predict(X_test_scaled)
accuracy_score(y_test , voting.predict(X_test_scaled))

0.8880597014925373

In [17]:
#stacking
stacking = StackingClassifier(
        estimators = [
        ("lr" , LogisticRegression(max_iter = 1000)),
        ("svm" , SVC()),
        ("dt" , DecisionTreeClassifier())
    ]
)
stacking.fit(X_train_scaled , y_train)
stacking.predict(X_test_scaled)
accuracy_score(y_test , stacking.predict(X_test_scaled))

0.914179104477612

In [18]:
#xgboost
xgb = XGBClassifier(
    n_estimators = 100,
    random_state = 42
)
xgb.fit(X_train_scaled , y_train)
xgb.predict(X_test_scaled)
accuracy_score(y_test , xgb.predict(X_test_scaled)
)

0.8917910447761194

In [19]:
# applying grid search cv to the top 3 models
# svm gridsearchcv
from sklearn.model_selection import GridSearchCV
param_grid = {
    "C" : [0.1 , 1 , 10],
    "kernel" : ["linear" , "poly" ,"rbf" , "sigmoid"],
    "gamma" : ["scale" , "auto"]
}
grid_svm = GridSearchCV(SVC() , param_grid , cv = 5 , scoring = "accuracy" , verbose = 1)
grid_svm.fit(X_train_scaled , y_train)
print("Best Params SVM:", grid_svm.best_params_)
print("Best Score SVM:", grid_svm.best_score_)
print("Test Accuracy:", accuracy_score(y_test, grid_svm.predict(X_test_scaled)))

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Best Params SVM: {'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best Score SVM: 0.8990654205607477
Test Accuracy: 0.9029850746268657


In [23]:
#gradientbosting gridsearchcv
param_grid_gb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5]
}

grid_gb = GridSearchCV(GradientBoostingClassifier(), param_grid_gb, cv=5, scoring="accuracy")
grid_gb.fit(X_train_scaled, y_train)
print("Best Params GB:", grid_gb.best_params_)
print("Best Score GB:", grid_gb.best_score_)
print("Test Accuracy:", accuracy_score(y_test, grid_gb.predict(X_test_scaled)))

Best Params GB: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Best Score GB: 0.9018691588785048
Test Accuracy: 0.9029850746268657


In [ ]:
import pickle
pickle.dump(grid_svm, open("model.pkl", "wb"))

In [1]:
!pip install streamlit